In [ ]:
# Routing in langgraph is the ability to conditionally determine which node to execute next 
# based on current state or the output of the node.
# It is typically implemented using: 
# a) add_conditional_edges:It maps node output to different possible next nodes
# b) State: The workflow's state can store variables that can influence routing decisions.
# c) Condition functions: Function that evaluates state or node output to decide next step.

# Key concepts:
# 1) Dynamic flow: Unlike linear sequence,routing let's the graph adapt to intermediate results. 
# 2) Conditional logic: You define the rules. 
# 3) Flexibility: Combines well with parallelization or sequential chains for complex workflows.

 

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model = "openai/gpt-oss-20b")
result = llm.invoke("Hello")


In [ ]:
from typing_extensions import Literal
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,SystemMessage
from typing_extensions import TypedDict
# Schema for structured output to use a routing logic
class Route(BaseModel):
    step:Literal["poem","story","joke"] = Field(description="The next step in the routing process")

# Augment the LLM with schema for structured output 
router = llm.with_structured_output(Route)

# State
class State(TypedDict):
    input:str
    decision:str
    output:str

def llm_call_1(state:State):
    """Write a story"""
    result = llm.invoke(state["input"])
    return {"output":result.content}

def llm_call_2(state:State):
    """Write a joke"""
    print("LLM Call 2 is called")
    result = llm.invoke(state["input"])
    return {"output":result.content}

def llm_call_3(state:State):
    """Write a poem"""
    print("LLM Call 3 is called")
    result = llm.invoke(state["input"])
    return {"output":result.content}

def llm_call_router(state:State):
    """Route the input to the appropriate node."""
    decision = router.invoke(
        [
            SystemMessage(content="Route the input to the story, joke or poem based on the user request"),
            HumanMessage(content=state["input"]),
        ]
    )
    return {"decision":decision.step}

def route_decision(state:State):
    if state["decision"] == "story":
        return "llm_call_1"
    if state["decision"] == "joke":
        return "llm_call_2"
    if state["decision"] == "poem":
        return "llm_call_3"
    raise ValueError(f"Unknown route: {state['decision']}")

from langgraph.graph import StateGraph,START,END
from IPython.display import Image,display
#Add nodes
router_builder = StateGraph(State)
router_builder.add_node("llm_call_1",llm_call_1)
router_builder.add_node("llm_call_2",llm_call_2)
router_builder.add_node("llm_call_3",llm_call_3)
router_builder.add_node("llm_call_router",llm_call_router)

from langgraph.graph import START,END,StateGraph
from IPython.display import Image,display
router_builder = StateGraph(State)
# Add nodes 
router_builder.add_node("llm_call_1",llm_call_1)
router_builder.add_node("llm_call_2",llm_call_2)
router_builder.add_node("llm_call_3",llm_call_3)
router_builder.add_node("llm_call_router",llm_call_router)

# Add edges 
router_builder.add_edge(START,"llm_call_router")
router_builder.add_conditional_edges(
    "llm_call_router",
    route_decision,
    {
        "llm_call_1":"llm_call_1",
        "llm_call_2":"llm_call_2",
        "llm_call_3":"llm_call_3"
    }
)
router_builder.add_edge("llm_call_1",END)
router_builder.add_edge("llm_call_2",END)
router_builder.add_edge("llm_call_3",END)

router_workflow = router_builder.compile()
display(Image(router_workflow.get_graph().draw_mermaid_png()))






In [ ]:
state = router_workflow.invoke({"input":"Write me a joke about Board exams"})
print(state["output"])